# Reframe frequency as classification — reproduce §14.14 (issue #67)

**What this reproduces:** the current branch's experiment — Spanish motor claim counts
(`N_claims_year`) re-expressed as **binary** (claim vs no-claim, 11.1% positive) and **ordinal**
(0/1/2+), scored on the multiclass-native metric set. Protocol: 5-fold, seeds 42 + 7 (binary),
seed 42 (ordinal), 9 methods (binary) / 6 methods (ordinal), leak-fixed load, TabPFN
`model_path="v3_default"` with the canonical retry loop.

**How it works:** calls the exact script that produced the committed results —
`scripts/eval/insurance_benchmark_v1/run_reframe_frequency.py` (no CLI args by design).

**Requirements:** benchmark-venv kernel + one-click **browser login** in the preflight cell (token saved to the gitignored repo-root `.env`), or `TABPFN_API_KEY` set manually — see `notebooks/reproducibility/README.md`.

**Cost:** ~120 fold-rows total; the hosted-API TabPFN arms dominate. Budget 20–60 minutes.

**⚠ Warning:** re-running **overwrites the committed** `reframe_frequency_results.csv`
(and the summary CSV if the analysis cell is run). For exploration, copy the CSVs aside first.

**Expected verdict (§14.14):** TabPFN ranks #1 on every metric, paired-significant and seed-stable
(binary AUC 0.7170, +0.0080 vs LightGBM, p=0.0010) — while the same rows scored as a count model
had TabPFN *behind* LGBM on Poisson deviance (§14.9). The count axis was the loss; classification is the win.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *[Path.cwd().parents[i] for i in range(1, 4)]]
            if (p / "scripts/eval/insurance_benchmark_v1/run_reframe_frequency.py").exists())
print("repo root:", ROOT)

In [ ]:
# Preflight: versions + auth. No TABPFN_API_KEY configured? One-click browser login:
# opens the Prior Labs login page, then saves the token to the repo-root .env
# (gitignored) and exports it for this session's subprocesses.
import importlib, os
for m in ("tabpfn_client", "pandas", "sklearn"):
    mod = importlib.import_module(m)
    print(f"{m:14s} {getattr(mod, '__version__', '?')}")

def _have_key() -> bool:
    if os.environ.get("TABPFN_API_KEY"):
        return True
    env = ROOT / ".env"
    return env.exists() and any(l.startswith("TABPFN_API_KEY=") for l in env.read_text().splitlines())

if not _have_key():
    from tabpfn_client.browser_auth import BrowserAuthHandler
    ok, token = BrowserAuthHandler().try_browser_login()
    assert ok and token, "Browser login failed — see notebooks/reproducibility/README.md"
    env = ROOT / ".env"
    lines = [l for l in env.read_text().splitlines() if not l.startswith("TABPFN_API_KEY=")] if env.exists() else []
    lines.append(f"TABPFN_API_KEY={token}")
    env.write_text("\n".join(lines) + "\n")
    os.environ["TABPFN_API_KEY"] = token
    print("Browser login OK — token saved to repo-root .env (gitignored).")
print("API key present:", _have_key())
assert _have_key(), "no API key — rerun this cell to trigger browser login, or set TABPFN_API_KEY manually"

In [ ]:
# The exact command behind §14.14 (no args — the protocol is hardcoded in the script).
import subprocess, time

cmd = [sys.executable, "scripts/eval/insurance_benchmark_v1/run_reframe_frequency.py"]
print("running:", " ".join(cmd))
t0 = time.time()
r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-5000:])
print(f"exit: {r.returncode}  ({time.time() - t0:.0f}s)")
if r.returncode:
    print(r.stderr[-2000:])

In [ ]:
# The evidence: 120 fold-rows (5 folds × (9 methods × 2 binary seeds + 6 methods × 1 ordinal seed)).
import pandas as pd
res = pd.read_csv(ROOT / "scripts/eval/insurance_benchmark_v1/reframe_frequency_results.csv")
print("rows:", len(res), "| tasks:", res.task.unique(), "| seeds:", sorted(res.seed.unique()))
res.head(3).T

In [ ]:
# Binary task — mean over folds, per method and seed. TabPFN should top AUC (and PR-AUC) both seeds.
bin_cols = ["log_loss", "auc", "pr_auc", "brier", "lift10"]
res[res.task == "binary"].groupby(["method", "seed"])[bin_cols].mean().round(4).sort_values("auc", ascending=False)

In [ ]:
# Ordinal task — one-vs-rest macro AUC, multiclass log loss / Brier, lift10 on P(>=1).
# PR-AUC is NaN by design (no standard multiclass PR-AUC) — lift10 is the substitute.
ord_cols = ["auc", "log_loss", "brier", "lift10"]
res[res.task == "ordinal"].groupby("method")[ord_cols].mean().round(4).sort_values("auc", ascending=False)

In [ ]:
# Optional: the paired-stats layer (deltas + p-values vs best GLM) that the report quotes.
# Re-running this also overwrites reframe_frequency_summary.csv.
r2 = subprocess.run([sys.executable, "scripts/eval/insurance_benchmark_v1/analyze_reframe_frequency.py"],
                    cwd=ROOT, capture_output=True, text=True)
print(r2.stdout[-4000:])
if r2.returncode:
    print(r2.stderr[-2000:])

## Reading the verdict

- **Binary (seed 42):** TabPFN AUC 0.7170 (+0.0080 vs LightGBM, p=0.0010); rank #1 on all five
  metrics. Seed 7 confirms (0.7165, +0.0098, p=0.0020).
- **Ordinal:** one-vs-rest AUC 0.7167 (+0.0111 vs LightGBM, p=0.0085) — the largest edge of the two reframes.
- **GLM collapse:** poissonglm/tweedieglm predict a constant on claim/no-claim (AUC exactly 0.5000,
  log loss at base rate 0.3490) — their count-domain edge does not transfer.
- **Scoping:** single dataset (spanish_motor_freq); freMTPL2freq (678K rows) was NOT reframed;
  the count model itself (Poisson deviance) and the pricing story are unchanged.

Master report §14.14; learning path S5 / Stage 4.5; digest `docs/MASTER-REPORT-DIGEST.md`.